<style>
    /* Crisp Light Clinical Theme */
    h1 { color: #0F172A; font-family: 'Inter', sans-serif; font-weight: 800; border-bottom: 3px solid #2563EB; padding-bottom: 10px; }
    h2 { color: #1E3A8A; font-family: 'Inter', sans-serif; font-weight: 700; margin-top: 25px; }
    h3 { color: #3B82F6; font-family: 'Inter', sans-serif; font-weight: 600; }
    .alert-info { background-color: #EFF6FF; color: #1D4ED8; border-left: 5px solid #2563EB; padding: 15px; border-radius: 4px; font-family: 'Inter', sans-serif; }
    .alert-warning { background-color: #FEF2F2; color: #B91C1C; border-left: 5px solid #DC2626; padding: 15px; border-radius: 4px; font-family: 'Inter', sans-serif; }
    .highlight { background-color: #F8FAFC; border: 1px solid #E2E8F0; padding: 10px; border-radius: 5px; font-family: monospace; color: #334155; }
</style>


<h1>🌍 Global Health Intelligence: API Extraction & ETL Pipeline</h1>
<p style="font-size: 1.1em; color: #475569;">
<strong>Author:</strong> Ali Naderi | <strong>Role:</strong> Senior Data Engineer & Machine Learning Architect<br>
<strong>Objective:</strong> To engineer a production-ready Extract, Transform, and Load (ETL) pipeline harvesting live epidemiological data from the World Health Organization (WHO) Global Health Observatory (GHO) OData API.
</p>

<div class="alert-info">
<strong>Executive Summary:</strong><br>
In modern Data Science, model accuracy is heavily constrained by data quality. This notebook demonstrates how to interface with complex, nested JSON REST APIs (like WHO GHO), handle network instability (timeouts/retries), perform strict schema validation, and load the optimized data into a relational SQL database. This ensures high data integrity and zero data leakage for downstream BI Dashboards and ML pipelines.
</div>


In [1]:
import requests
import sqlite3
import pandas as pd
import logging
import time

# Configure professional logging instead of raw prints
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


<h2>1. The "Extract" Phase: Robust API Communication</h2>
<p>
Why use exponential backoff and error handling? Real-world APIs impose rate limits and frequently drop connections. A naive <code>requests.get()</code> is unsuitable for production. We implement a robust extraction layer that ensures data is retrieved reliably without overwhelming the WHO servers.
</p>


In [2]:
def extract_who_data(indicator_code: str, max_retries: int = 3) -> list:
    '''
    Extracts JSON records from the WHO GHO OData API using robust retry logic.
    '''
    url = f"https://ghoapi.azureedge.net/api/{indicator_code}"
    
    for attempt in range(max_retries):
        try:
            logger.info(f"Connecting to WHO API for indicator: {indicator_code} (Attempt {attempt+1})")
            response = requests.get(url, timeout=30)
            response.raise_for_status() # Catch HTTP 4xx/5xx errors
            
            data = response.json()
            records = data.get('value', [])
            logger.info(f"Successfully extracted {len(records)} records.")
            return records
            
        except requests.exceptions.RequestException as e:
            logger.error(f"Network error: {e}")
            if attempt < max_retries - 1:
                sleep_time = 2 ** attempt
                logger.info(f"Backing off for {sleep_time} seconds before retrying...")
                time.sleep(sleep_time)
            else:
                logger.error("Max retries exceeded. Extraction failed.")
                return []
                
# Test the extraction on Life Expectancy
raw_life_expectancy = extract_who_data("WHOSIS_000001")
print(f"Sample Record: {raw_life_expectancy[0] if raw_life_expectancy else 'No data'}")


2026-07-16 08:57:43,699 - INFO - Connecting to WHO API for indicator: WHOSIS_000001 (Attempt 1)
2026-07-16 08:58:02,178 - INFO - Successfully extracted 12936 records.


Sample Record: {'Id': 6951992, 'IndicatorCode': 'WHOSIS_000001', 'SpatialDimType': 'COUNTRY', 'SpatialDim': 'GTM', 'TimeDimType': 'YEAR', 'ParentLocationCode': 'AMR', 'ParentLocation': 'Americas', 'Dim1Type': 'SEX', 'Dim1': 'SEX_BTSX', 'TimeDim': 2006, 'Dim2Type': None, 'Dim2': None, 'Dim3Type': None, 'Dim3': None, 'DataSourceDimType': None, 'DataSourceDim': None, 'Value': '69.9 [69.5-70.2]', 'NumericValue': 69.85634314, 'Low': 69.52250836, 'High': 70.18470182, 'Comments': None, 'Date': '2024-08-02T09:43:39.193+02:00', 'TimeDimensionValue': '2006', 'TimeDimensionBegin': '2006-01-01T00:00:00+01:00', 'TimeDimensionEnd': '2006-12-31T00:00:00+01:00'}


<h2>2. The "Transform" Phase: Data Integrity & Memory Optimization</h2>
<p>
Raw API data is often nested, contains redundant fields, and uses inefficient string data types. Here, we transform the JSON array into a Pandas DataFrame.
</p>
<div class="alert-warning">
<strong>Why Memory Optimization Matters:</strong><br>
When dealing with Big Data, storing country codes or indicators as standard python strings consumes massive RAM. By downcasting strings to <code>pd.Categorical</code> and numerics to <code>float32</code>/<code>int32</code>, we can reduce the memory footprint by up to <strong>70%</strong>, allowing the pipeline to run efficiently on standard servers.
</div>


In [3]:
def transform_who_data(records: list, indicator_name: str) -> pd.DataFrame:
    '''
    Cleans and optimizes the raw WHO JSON records.
    '''
    if not records:
        return pd.DataFrame()
        
    df = pd.DataFrame(records)
    
    # 1. Feature Selection & Renaming (Standardizing the Schema)
    df = df[['SpatialDim', 'TimeDim', 'Dim1', 'NumericValue']].rename(columns={
        'SpatialDim': 'CountryCode',
        'TimeDim': 'Year',
        'Dim1': 'Gender',
        'NumericValue': 'Value'
    })
    
    # 2. Handling Missing Data (Data Integrity)
    df['Indicator'] = indicator_name
    df = df.dropna(subset=['Value', 'CountryCode', 'Year'])
    df['Gender'] = df['Gender'].fillna('Total')
    
    # 3. Memory Optimization (Downcasting types)
    memory_before = df.memory_usage(deep=True).sum() / 1024**2
    
    df['CountryCode'] = df['CountryCode'].astype('category')
    df['Gender'] = df['Gender'].astype('category')
    df['Indicator'] = df['Indicator'].astype('category')
    df['Year'] = df['Year'].astype('int32')
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce').astype('float32')
    df = df.dropna(subset=['Value']) # Drop rows where value couldn't be parsed
    
    memory_after = df.memory_usage(deep=True).sum() / 1024**2
    
    logger.info(f"Memory footprint reduced from {memory_before:.2f} MB to {memory_after:.2f} MB")
    
    return df

clean_life_exp_df = transform_who_data(raw_life_expectancy, "Life_Expectancy")
clean_life_exp_df.head()


2026-07-16 08:58:10,559 - INFO - Memory footprint reduced from 0.81 MB to 0.15 MB


,CountryCode,Year,Gender,Value,Indicator
0,GTM,2006,SEX_BTSX,69.856346,Life_Expectancy
1,CAN,2017,SEX_BTSX,81.709541,Life_Expectancy
2,NLD,2015,SEX_BTSX,81.280693,Life_Expectancy
3,AFR,2008,SEX_FMLE,59.583523,Life_Expectancy
4,CUB,2021,SEX_MLE,71.125954,Life_Expectancy


<h2>3. The "Load" Phase: Storing in a Relational Database</h2>
<p>
Data Analysts and BI tools expect structured SQL databases, not raw CSV files. Loading the optimized DataFrame into an <code>SQLite</code> database ensures data persists safely and can be queried instantly by our Streamlit BI Dashboard without reloading the entire dataset into memory.
</p>


In [4]:
def load_to_sql(df: pd.DataFrame, db_path: str = "who_data.db"):
    '''
    Persists the cleaned DataFrame to a local SQLite Database.
    '''
    if df.empty:
        logger.warning("No data to load.")
        return
        
    try:
        # Using context manager ensures the connection closes safely
        with sqlite3.connect(db_path) as conn:
            # We use if_exists='append' to add to existing tables
            df.to_sql('health_indicators', conn, if_exists='replace', index=False)
            logger.info(f"Successfully loaded {len(df)} rows into {db_path}.")
    except Exception as e:
        logger.error(f"Database error: {e}")

load_to_sql(clean_life_exp_df)


2026-07-16 08:58:37,109 - INFO - Successfully loaded 12936 rows into who_data.db.


<div class="alert-info">
<strong>Conclusion & Next Steps:</strong><br>
We have successfully engineered a robust, memory-efficient ETL pipeline. The data is now securely stored in <code>who_data.db</code>. The next step is to launch the interactive BI Dashboard (<code>app.py</code>) using Streamlit, which will query this database via <code>st.cache_data</code> to render live, interactive choropleth maps and time-series analytics.
</div>
